# Week-6 Spark Assignment
### Step 1: Run this cell first (Setup)

In [1]:
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName('Week6').getOrCreate()
print('Spark ready! Version:', spark.version)

Spark ready! Version: 4.0.3


---
## Q1: Driver, Cluster Manager, and Executor Roles

In [2]:
print("""
DRIVER:
  - Main program that controls the Spark application
  - Creates SparkContext/SparkSession
  - Builds the DAG (execution plan)
  - Distributes tasks to Executors

CLUSTER MANAGER:
  - Allocates resources (CPU, Memory)
  - Types: YARN, Mesos, Kubernetes, Standalone
  - Acts as coordinator between Driver and Executors

EXECUTOR:
  - Runs on worker nodes
  - Executes the actual tasks
  - Stores data in memory/disk
  - Returns results to the Driver
""")


DRIVER:
  - Main program that controls the Spark application
  - Creates SparkContext/SparkSession
  - Builds the DAG (execution plan)
  - Distributes tasks to Executors

CLUSTER MANAGER:
  - Allocates resources (CPU, Memory)
  - Types: YARN, Mesos, Kubernetes, Standalone
  - Acts as coordinator between Driver and Executors

EXECUTOR:
  - Runs on worker nodes
  - Executes the actual tasks
  - Stores data in memory/disk
  - Returns results to the Driver



---
## Q2: Lazy Evaluation — Performance Demo

In [3]:
data = [(1,'Alice',30),(2,'Bob',25),(3,'Charlie',35)]
df = spark.createDataFrame(data, ['id','name','age'])

# These are LAZY — they will not execute yet
filtered = df.filter(col('age') > 28)     # Transformation 1
selected = filtered.select('name','age')   # Transformation 2
print('No execution has happened yet (Lazy Evaluation)!')
print('Will execute only when an Action is called:')

# ACTION — now it executes with an optimized plan
selected.show()

No execution has happened yet (Lazy Evaluation)!
Will execute only when an Action is called:
+-------+---+
|   name|age|
+-------+---+
|  Alice| 30|
|Charlie| 35|
+-------+---+



---
## Q3: Reading a CSV File

In [4]:
import os
os.makedirs('data', exist_ok=True)

# Create a sample CSV file
with open('data/source.csv','w') as f:
    f.write('product_id,product_name,price,category\n')
    f.write('101,Laptop,75000,Electronics\n')
    f.write('102,Phone,25000,Electronics\n')
    f.write('103,Shirt,1500,Clothing\n')
    f.write('104,TV,45000,Electronics\n')

# Q3 ANSWER:
df = spark.read.option('header', True).option('inferSchema', True).csv('data/source.csv')
df.printSchema()
df.show()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- category: string (nullable = true)

+----------+------------+-----+-----------+
|product_id|product_name|price|   category|
+----------+------------+-----+-----------+
|       101|      Laptop|75000|Electronics|
|       102|       Phone|25000|Electronics|
|       103|       Shirt| 1500|   Clothing|
|       104|          TV|45000|Electronics|
+----------+------------+-----+-----------+



---
## Q4: CSV vs Parquet

In [5]:
print("""
CSV (Row-based):
  - Data is stored row by row
  - Human readable, uses more storage
  - Queries are slow (entire row must be read)

Parquet (Columnar):
  - Data is stored column by column
  - Compressed binary format, uses less storage
  - Queries are fast (only needed columns are loaded)
  - Has a built-in schema

Example: Only 2 columns needed out of 100:
  CSV     -> reads the entire row (slow)
  Parquet -> loads only 2 columns (fast!)
""")

df.write.mode('overwrite').parquet('data/source.parquet')
print('Parquet file saved!')


CSV (Row-based):
  - Data is stored row by row
  - Human readable, uses more storage
  - Queries are slow (entire row must be read)

Parquet (Columnar):
  - Data is stored column by column
  - Compressed binary format, uses less storage
  - Queries are fast (only needed columns are loaded)
  - Has a built-in schema

Example: Only 2 columns needed out of 100:
  CSV     -> reads the entire row (slow)
  Parquet -> loads only 2 columns (fast!)

Parquet file saved!


---
## Q5: Filter Electronics + Select Columns

In [6]:
# Select product_id and price where category = 'Electronics'
df.filter(df['category'] == 'Electronics').select('product_id','price').show()

+----------+-----+
|product_id|price|
+----------+-----+
|       101|75000|
|       102|25000|
|       104|45000|
+----------+-----+



---
## Q6: Column Rename + Type Cast

In [7]:
data2 = [('P01','Laptop','75000.50'),('P02','Phone','25000.00')]
df2 = spark.createDataFrame(data2, ['old_name','product','price'])

print('Before:'); df2.printSchema()

df2 = df2.withColumnRenamed('old_name', 'new_name') \
         .withColumn('price', col('price').cast('Double'))

print('After:'); df2.printSchema(); df2.show()

Before:
root
 |-- old_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- price: string (nullable = true)

After:
root
 |-- new_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- price: double (nullable = true)

+--------+-------+-------+
|new_name|product|  price|
+--------+-------+-------+
|     P01| Laptop|75000.5|
|     P02|  Phone|25000.0|
+--------+-------+-------+



---
## Q7: Lineage Graph (DAG) and Fault Tolerance

In [8]:
print("""
What is a Lineage Graph (DAG):
  - Spark records every transformation applied
  - This record is a DAG (Directed Acyclic Graph)

Fault Tolerance:
  - If a worker node fails -> only the lost partition is recomputed
  - The entire file does not need to be re-read
  - Recovery is automatic
""")

# View the execution plan / DAG
df3 = df.filter(col('price') > 10000).select('product_name','price')
print('DAG (Execution Plan):')
df3.explain()


What is a Lineage Graph (DAG):
  - Spark records every transformation applied
  - This record is a DAG (Directed Acyclic Graph)

Fault Tolerance:
  - If a worker node fails -> only the lost partition is recomputed
  - The entire file does not need to be re-read
  - Recovery is automatic

DAG (Execution Plan):
== Physical Plan ==
*(1) Filter (isnotnull(price#29) AND (price#29 > 10000))
+- FileScan csv [product_name#28,price#29] Batched: false, DataFilters: [isnotnull(price#29), (price#29 > 10000)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/data/source.csv], PartitionFilters: [], PushedFilters: [IsNotNull(price), GreaterThan(price,10000)], ReadSchema: struct<product_name:string,price:int>




---
## Q8: Status = 'Completed' AND Amount > 1000

In [9]:
orders = [(1,'Completed',1500),(2,'Pending',2000),(3,'Completed',800),(4,'Completed',3000),(5,'Cancelled',1200)]
df_orders = spark.createDataFrame(orders, ['order_id','status','amount'])

print('All Orders:'); df_orders.show()

print('Completed AND amount > 1000:')
df_orders.filter((df_orders['status']=='Completed') & (df_orders['amount']>1000)).show()

All Orders:
+--------+---------+------+
|order_id|   status|amount|
+--------+---------+------+
|       1|Completed|  1500|
|       2|  Pending|  2000|
|       3|Completed|   800|
|       4|Completed|  3000|
|       5|Cancelled|  1200|
+--------+---------+------+

Completed AND amount > 1000:
+--------+---------+------+
|order_id|   status|amount|
+--------+---------+------+
|       1|Completed|  1500|
|       4|Completed|  3000|
+--------+---------+------+



---
## Q9: Predicate Pushdown in Parquet

In [10]:
print("""
Predicate Pushdown:
  - The filter (WHERE price > 500) is applied while reading the file
  - Only relevant data is loaded into memory

Without Pushdown: Load everything -> then filter (slow)
With Pushdown:    Filter first -> load only matching data (fast!)

This is not possible with CSV — entire file must be read
""")

df_p = spark.read.parquet('data/source.parquet')
print('Check PushedFilters in the plan below:')
df_p.filter(col('price') > 30000).explain()


Predicate Pushdown:
  - The filter (WHERE price > 500) is applied while reading the file
  - Only relevant data is loaded into memory

Without Pushdown: Load everything -> then filter (slow)
With Pushdown:    Filter first -> load only matching data (fast!)

This is not possible with CSV — entire file must be read

Check PushedFilters in the plan below:
== Physical Plan ==
*(1) Filter (isnotnull(price#103) AND (price#103 > 30000))
+- *(1) ColumnarToRow
   +- FileScan parquet [product_id#101,product_name#102,price#103,category#104] Batched: true, DataFilters: [isnotnull(price#103), (price#103 > 30000)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/data/source.parquet], PartitionFilters: [], PushedFilters: [IsNotNull(price), GreaterThan(price,30000)], ReadSchema: struct<product_id:int,product_name:string,price:int,category:string>




---
## Q10: Add final_price Column with 18% Tax

In [11]:
price_data = [(101,'Laptop',75000.0),(102,'Phone',25000.0),(103,'TV',45000.0)]
df_price = spark.createDataFrame(price_data, ['id','product','base_price'])

df_price = df_price.withColumn('final_price', col('base_price') * 1.18)

print('final_price with 18% Tax:')
df_price.show()

final_price with 18% Tax:
+---+-------+----------+-----------+
| id|product|base_price|final_price|
+---+-------+----------+-----------+
|101| Laptop|   75000.0|    88500.0|
|102|  Phone|   25000.0|    29500.0|
|103|     TV|   45000.0|    53100.0|
+---+-------+----------+-----------+



---
## Q11: Transformations vs Actions

In [12]:
sample = [(1,'A',100),(2,'B',200),(3,'C',300),(4,'D',400)]
df_s = spark.createDataFrame(sample, ['id','name','value'])

# TRANSFORMATIONS (Lazy — not executed yet)
t1 = df_s.filter(col('value') > 150)           # Transformation 1
t2 = df_s.select('name','value')               # Transformation 2
t3 = df_s.withColumn('double', col('value')*2) # Transformation 3
t4 = df_s.groupBy('name').count()              # Transformation 4
print('Transformations defined (not executed yet)...')

# ACTIONS (trigger actual execution)
print('\nACTION 1 - show():'); t2.show()
print('ACTION 2 - count():', df_s.count())
print('ACTION 3 - first():', df_s.first())
print('ACTION 4 - collect() first 2:', df_s.collect()[:2])

Transformations defined (not executed yet)...

ACTION 1 - show():
+----+-----+
|name|value|
+----+-----+
|   A|  100|
|   B|  200|
|   C|  300|
|   D|  400|
+----+-----+

ACTION 2 - count(): 4
ACTION 3 - first(): Row(id=1, name='A', value=100)
ACTION 4 - collect() first 2: [Row(id=1, name='A', value=100), Row(id=2, name='B', value=200)]


---
## Q12: Parquet Load → Filter Nulls → Save as CSV

In [13]:
# Create sample data with nulls
null_data = [(1,'Alice'),(2,None),(3,'Charlie'),(None,'Dave')]
df_null = spark.createDataFrame(null_data, ['user_id','name'])
df_null.write.mode('overwrite').parquet('path/to/input')

# Q12 ANSWER:
spark.read.parquet('path/to/input') \
     .filter(col('user_id').isNotNull()) \
     .write.mode('overwrite').option('header', True).csv('path/to/output')

# Verify result
result = spark.read.option('header', True).csv('path/to/output')
print('Null rows removed:')
result.show()

Null rows removed:
+-------+-------+
|user_id|   name|
+-------+-------+
|      1|  Alice|
|      2|   NULL|
|      3|Charlie|
+-------+-------+



---
## Q13: Client Mode vs Cluster Mode

In [14]:
print("""
CLIENT MODE:
  - Driver runs on the local machine (client)
  - Client must stay connected throughout
  - Used for development and testing
  - Command: spark-submit --deploy-mode client job.py

CLUSTER MODE:
  - Driver runs on a node inside the cluster
  - Client can disconnect (job keeps running)
  - Used for production workloads
  - Command: spark-submit --deploy-mode cluster job.py
""")


CLIENT MODE:
  - Driver runs on the local machine (client)
  - Client must stay connected throughout
  - Used for development and testing
  - Command: spark-submit --deploy-mode client job.py

CLUSTER MODE:
  - Driver runs on a node inside the cluster
  - Client can disconnect (job keeps running)
  - Used for production workloads
  - Command: spark-submit --deploy-mode cluster job.py



---
## Q14: Region = 'North' OR Priority = 'High'

In [15]:
region_data = [(1,'North','Low'),(2,'South','High'),(3,'East','Medium'),(4,'North','High'),(5,'West','Low')]
df_r = spark.createDataFrame(region_data, ['id','region','priority'])

print('All Data:'); df_r.show()

print("Region = North OR Priority = High:")
df_r.filter((df_r['region']=='North') | (df_r['priority']=='High')).show()

All Data:
+---+------+--------+
| id|region|priority|
+---+------+--------+
|  1| North|     Low|
|  2| South|    High|
|  3|  East|  Medium|
|  4| North|    High|
|  5|  West|     Low|
+---+------+--------+

Region = North OR Priority = High:
+---+------+--------+
| id|region|priority|
+---+------+--------+
|  1| North|     Low|
|  2| South|    High|
|  4| North|    High|
+---+------+--------+



---
## Q15: .show(5) vs .collect() on Large Datasets

In [16]:
print("""
.collect() — DANGEROUS on large data:
  - Loads ALL data into the Driver machine's memory
  - On terabytes of data = OutOfMemoryError / Crash

.show(5) — SAFE:
  - Fetches only the first 5 rows
  - Ignores the rest
  - Memory efficient and perfect for exploration
""")

print('SAFE - show(5):')
df.show(5)

print('collect() result (fine on small data):')
print(f'Total rows in memory: {len(df.collect())} — on large data this can crash!')


.collect() — DANGEROUS on large data:
  - Loads ALL data into the Driver machine's memory
  - On terabytes of data = OutOfMemoryError / Crash

.show(5) — SAFE:
  - Fetches only the first 5 rows
  - Ignores the rest
  - Memory efficient and perfect for exploration

SAFE - show(5):
+----------+------------+-----+-----------+
|product_id|product_name|price|   category|
+----------+------------+-----+-----------+
|       101|      Laptop|75000|Electronics|
|       102|       Phone|25000|Electronics|
|       103|       Shirt| 1500|   Clothing|
|       104|          TV|45000|Electronics|
+----------+------------+-----+-----------+

collect() result (fine on small data):
Total rows in memory: 4 — on large data this can crash!


---
## 📊 Brief Insights on Performance & Architecture

### 🏗️ Architecture Insights
- **Driver** is a single point of control — if the Driver crashes, the entire job fails
- **More Executors** = more parallel processing = faster job execution
- **Cluster Manager** (YARN/Kubernetes) dynamically allocates resources — idle resources are not wasted

### ⚡ Performance Insights
- **Lazy Evaluation** → Spark builds the full plan first, then executes it in one optimized pass — unnecessary steps are automatically removed
- **DAG (Lineage Graph)** → Fault tolerance comes for free; if a node fails, only the lost partition is recomputed, not the entire job
- **Predicate Pushdown (Parquet)** → Filters are applied at the data source level, less data enters memory = faster queries
- **Parquet vs CSV** → On large datasets, Parquet can be 5x–10x faster because columnar format reads only the required columns
- **Shuffling** → Operations like `groupBy` and `join` move data across the network (shuffle) — this is the most expensive operation in Spark; minimize it where possible

### 🛡️ Best Practices
- Always use `.show(5)` for exploration; never run `.collect()` on large datasets
- Filter null values early in the pipeline — doing it later is more costly
- Prefer Parquet over CSV for production pipelines
- Follow this pipeline order: **read → transform → filter → write**
